In [ ]:
import numpy as np


spend_cols = [
    col
    for col in wide_df.columns
    if col.startswith('spend_')
]

clean_df = wide_df.copy()

for col in spend_cols:

    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    clean_df = clean_df[
        clean_df[col].between(lower_bound, upper_bound)
    ]


print('До очистки:', len(wide_df))
print('После очистки:', len(clean_df))
print('Удалено:', len(wide_df) - len(clean_df))

In [ ]:
long_df = (
    wide_df
    .melt(
        id_vars=[
            'client_id',
            'campaigns_cnt',
            'first_campaign_dt'
        ],
        value_vars=spend_cols,
        var_name='period',
        value_name='month_spend'
    )
)

clean_parts = []

for (cohort, period), part in long_df.groupby(
    ['campaigns_cnt', 'period']
):

    q1 = part['month_spend'].quantile(0.25)
    q3 = part['month_spend'].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    cleaned = part[
        part['month_spend'].between(lower, upper)
    ]

    clean_parts.append(cleaned)

clean_long_df = pd.concat(clean_parts, ignore_index=True)

print('До очистки:', len(long_df))
print('После очистки:', len(clean_long_df))

In [ ]:
cohort_stats = (
    clean_long_df
    .groupby(
        ['campaigns_cnt', 'period'],
        as_index=False
    )
    .agg(
        avg_spend_per_client=('month_spend', 'mean'),
        median_spend_per_client=('month_spend', 'median'),
        total_spend=('month_spend', 'sum'),
        client_cnt=('client_id', 'nunique')
    )
)

cohort_stats.head()

In [ ]:
def period_order(period):

    if period == 'spend_m1':
        return -1

    if period == 'spend_0':
        return 0

    return int(period.replace('spend_p', ''))


cohort_stats['period_order'] = (
    cohort_stats['period']
    .apply(period_order)
)

cohort_stats = cohort_stats.sort_values(
    ['campaigns_cnt', 'period_order']
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


plt.figure(figsize=(14, 7))

for cohort in sorted(cohort_stats['campaigns_cnt'].unique()):

    part = cohort_stats[
        cohort_stats['campaigns_cnt'] == cohort
    ]

    plt.plot(
        part['period_order'],
        part['median_spend_per_client'],
        marker='o',
        linewidth=2,
        label=f'{cohort} камп.'
    )

plt.axvline(
    x=0,
    linestyle='--',
    alpha=0.5
)

plt.xticks(
    [-1, 0, 1, 2, 3, 4],
    [
        '-1 мес',
        '1-я камп.',
        '+1',
        '+2',
        '+3',
        '+4'
    ]
)

plt.title('Медианный РТО по когортам')
plt.xlabel('Месяц относительно первой кампании')
plt.ylabel('Медианный РТО')

plt.grid(True, alpha=0.3)

plt.legend(title='Кол-во кампаний')

plt.gca().yaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f'{x:,.0f}'.replace(',', ' ')
    )
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))

for cohort in sorted(cohort_stats['campaigns_cnt'].unique()):

    part = cohort_stats[
        cohort_stats['campaigns_cnt'] == cohort
    ]

    plt.plot(
        part['period_order'],
        part['avg_spend_per_client'],
        marker='o',
        linewidth=2,
        label=f'{cohort} камп.'
    )

plt.axvline(
    x=0,
    linestyle='--',
    alpha=0.5
)

plt.xticks(
    [-1, 0, 1, 2, 3, 4],
    [
        '-1 мес',
        '1-я камп.',
        '+1',
        '+2',
        '+3',
        '+4'
    ]
)

plt.title('Средний РТО по когортам')
plt.xlabel('Месяц относительно первой кампании')
plt.ylabel('Средний РТО')

plt.grid(True, alpha=0.3)

plt.legend(title='Кол-во кампаний')

plt.gca().yaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f'{x:,.0f}'.replace(',', ' ')
    )
)

plt.tight_layout()
plt.show()